# DSAN 6000 Homework 7: A Single Spark Cluster Can Light a Prairie Fire

As a reminder: the **code** within this notebook is provided for you, since the goal for this one is *just* to get a Spark cluster up and running, *not* to obtain proficiency with PySpark coding (that will come in the next HW!).

So, **your** main job is to **carry out the steps in the** [**Week 8 slides checklist**](https://jjacobs.me/dsan6000/w08/slides.html#/in-class-demo-2-your-first-spark-cluster) using (mainly) your **AWS Console**. Then, **once they have been run**, you will know that you've set up your cluster correctly if the Python code cells below "just work" as provided – if they don't, we have provided some hints in the instructions below, with some common error messages and what they mean. If you're still having trouble getting the code to run, feel free to post on the Google Space or [email us](mailto:dsan6000@georgetown.edu)!

## Setup Part 1: Launching the Master Node Process on *This* EC2 Instance

As shown in the checklist linked above, once you have **updated the Security Group settings** to allow incoming connections from instances within the Security Group, the next steps are to (a) install OpenJDK 21 and then (b) download the latest version of Spark using `wget`, into your **home** directory (the directory at path `~/`) on this EC2 instance, and (c) extract the `.tgz` file using `tar -xzvf <filename>.tgz`.

Once the Spark `.tgz` file has been extracted, you are set up to open the **Integrated Terminal** here in VSCode and navigate (using `cd`) into the `sbin` directory, where you should see the `start-master.sh` file – this is the key step!

However, you *shouldn not* run this `.sh` file without any arguments, as the default settings won't allow worker nodes to connect to the master process. Instead, you need to **find the *private* IP address** for this EC2 instance, either by copying it from the web-based **EC2 Console**, or by executing

```bash
hostname -I
```

within the Integrated Terminal.

Once you have this **Private IP address** for the EC2 instance, you will **start the master node process** to *listen for connections* to this IP address, by running `./start-master.sh` with the `-h` argument as follows:

```bash
./start-master.sh -h <Private IP>
```

Note that this command will *not* show a status, success/failure message, or... really anything useful in its output.

To check that it ran successfully, and to monitor what is happening on the master node, you will instead use the **web-based Spark master node UI**. If your `./start-master.sh` command executed as intended, this master node monitoring UI will be **running on your EC2 instance** on **port 8080**. To actually **see** this UI from your laptop, you will need to **forward port 8080** using the **"Ports"** tab in the bottom panel of the VSCode interface (in my VSCode setup, this is the tab directly to the right of the "Terminal" tab, and before the "Jupyter" tab).

If you are not currently forwarding any ports, you should see the text "No forwarded ports. Forward a port to access your running services locally", above a "Forward a Port" button. Click that button, then just type "8080" in the Port field that appears. You should then see "8080" under "Port", followed by "localhost:8080" under "Forwarded Address".

This means that, if all has been set up correctly, you should be able to **open your local browser**, type `localhost:8080` into the address bar, and see a web UI with **"Spark Master at spark://<Private IP>:7077"** at the top (with the Spark logo in the upper left corner), as in the following screenshot:

<center>

<img src='images/spark-master.jpeg' width='50%'>

</center>

Congratulations! This means that the master node process has been successfully launched on this EC2 instance, and you are now ready to **connect to it** from within this Jupyter notebook, and control it programmatically using `PySpark`, in the next part (Code Part 1).

## Code Part 1: Initializing a `SparkSession` Connected to the Master Process

Once you have your "main" EC2 instance set up as the **master node** for your Spark cluster, the following code cell then just sets up a `SparkSession` that "plugs into" this master node, thus allowing you to **control** your cluster programmatically using `PySpark` code.

Since much of the work here is **setting the environment up correcty** to allow the functioning of the Spark cluster, lots of things can go wrong when you run the following code cell, and that's ok! A core idea here is to **learn what the error messages are telling you** in terms of what is left to set up.

For example, if the `SparkSession` setup cell runs for ~a minute but then fails with the following message:

```
ERROR StandaloneSchedulerBackend: Application has been killed. Reason: All masters are unresponsive! Giving up.
```

This is telling you that the `SparkSession` was **unable to connect to the master node process** running on your instance. So, the issue is likely related to either (a) this instance's **Security Group** not allowing incoming connections from within the security group, or (b) the master node process (the execution of `./start-master.sh`) not running.

Note that, if you get that error, then try something to fix it, you will need to **restart this Jupyter notebook's kernel** before running the following code cell again. This is because of the fact mentioned in class that there can only be **one `SparkSession` per instance**. When the `getOrCreate()` function times out, and produces the above error message, it turns out that this means the `SparkSession` has been **stopped** but **not *terminated***. Although it is possible to terminate the `SparkSession`s programmatically, it makes for clearer code intent if you restart the kernel (since otherwise, someone reading your code might think that they *need* to terminate and restart it, when in reality if the master node is running correctly this is not necessary).

However, if the `SparkSession` **was** set up as intended, the output from running the following cell (because the last line just displays the `spark` variable) should look something like the following:

**SparkSession - in-memory**
                
**SparkContext**

Spark UI

<dl>
  <dt>Version</dt>
    <dd><code>v4.2.0</code></dd>
  <dt>Master</dt>
    <dd><code>spark://Private IP:7077</code></dd>
  <dt>AppName</dt>
    <dd><code>UNSC_Speeches</code></dd>
</dl>

Finally, note the additional line in the `SparkSession` initialization here that you haven't seen before:

```python
.config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
```

You can think of this line as the Spark "equivalent" of how you've used `import boto3` in previous assignments to interact with AWS services from within Python – it just "adds in" [additional functionality](https://mvnrepository.com/artifact/org.apache.hadoop/hadoop-aws/3.5.0) which will allow your Spark cluster to use S3 URIs like `s3a://my-s3-bucket/my-data.parquet` to work with data in **S3 buckets** the same way it works with "standard" standalone `.csv` or `.parquet` files.

In [ ]:
#| label: sparksession-init
from pyspark.sql import SparkSession

spark = (
    SparkSession
      .builder
      .appName('UNSC_Speeches')
      .master('spark://172.31.44.247:7077')
      .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
      .getOrCreate()
)
spark

As a final "sanity check" that the `SparkSession` successfully connected to the master node process, the following code cell will print out the URI for the master node. This is a weird address, since it *looks* like a URL, but in fact it's an "internal" address that Spark nodes use to communicate with each other:

In [5]:
#| label: print-master-uri
print(spark.sparkContext.master)

spark://172.31.44.247:7077


## Setup Part 2: Initializing and Connecting a Worker Node

Setting up the master node is great (it's at the core of what people mean by "launching" or "spinning up" a Spark cluster), but it can't do much on its own – we now need to set up **worker nodes**, and connect them to the master process running on this EC2 instance, so that Spark knows it can **delegate** computational tasks to these nodes.

Setting up the worker nodes as you'll do in this step is also good practice for getting comfortable quickly **creating** and **interacting with** "ephemeral" EC2 instances. I say "ephemeral" here specifically in contrast to the mental model you may have used for **this** EC2 instance: so far you've been using this instance essentially like a "remote computer" that you can log into to work on DSAN 6000 homeworks. The instances you'll create to serve as **worker nodes**, however, will instead be temporary computer instances that you create *solely* to carry out some particularly computationally-intensive task. Once they have finished working on that task, you will *terminate* them!

So, for this second setup step, your job is to use the **AWS Console** to create **three additional EC2 instances**, called `worker-1`, `worker-2`, and `worker-3`, where (as shown in class) the key **new step** when creating these instances will be **ensuring that they are in the same Security Group as the master node instance**. If you set up this EC2 instance using the default settings for new EC2 instances in the AWS Console, then its security group will be called `launch-wizard-1`.

All other settings in the creation of the new instances are the same as before: Ubuntu for the OS, `t3.large` for the type, 30 GB for the root disk, and `LabInstanceProfile` for the IAM profile.

For each of these three new instances, once AWS finishes creating them, your task will then be to **connect to** the instance and carry out the steps in the checklist. The new thing here is that **you do *not* need to launch three new VSCode sessions** to connect to these instances. Instead, you can build your familiarity with the in-browser AWS shell!

Within the **EC2 Console**, single-click on an existing instance (say, `worker-1`) within the list of instances, and you should see a "Connect" button above the list. Click that button, then confirm the default settings on the next page (there should be another orange "Connect" button in the bottom right of this screen), and you should see an **interactive terminal** that you can use to carry out the steps in the checklist. As a quick summary:

1.  Update **Ubuntu** itself using `sudo apt update`
2.  Install OpenJDK 21, using `sudo apt install`
3.  Download the latest Apache `.tgz` file using `wget`
4.  Extract the contents of the `.tgz` file using `tar -xzvf

Once these steps have been completed, you should have an **extracted folder** within the home directory of the instance. Within this extracted folder, navigate to the `sbin` subdirectory (using `cd`), where you should now see a `start-worker.sh` script. As was the case for the master node, you do *not* want to run this with the default settings. Instead, you should include a **single additional argument** after `./start-worker.sh`, that is just **the private IP address of the master node that you found above!** The command that you execute on the worker instances should thus look like:

```bash
./start-worker.sh <Master Node Private IP>:7077
```

Once you have connected **one** worker node in this way, you should **monitor** that the connections indeed worked. Keep the master node UI open at `localhost:8080` on your local laptop (the forwarded port): you will know that the connection from worker to master node has been established successfully when a new row appears in the "Worker" section, as in the following screenshot:

<center>

<img src='images/spark-firstworker.jpeg' width='50%'>

</center>

Once you have connected **all three** worker nodes to the master node, you can come back here and start running the code in the following section!

## Code Part 2: "Pointing" Your `SparkSession` at the Data

Now that you've connected the worker node, and you've verified that 2 cores are **available** to the cluster, it's time to see what this "availability" of cores actually **means** in terms of how Spark operates on data. It will be easier to see what's going on "under the hood" in Spark if we now "point" this cluster at an actual **dataset**. That way, rather than just monitoring the available nodes within the cluster in the abstract using the UI, we can concretely **check how many partitions** Spark has "chopped" the full dataset into at any given point.

In other words, by indicating to Spark the dataset we'd like to work with at this point, we can see the actual **work** that Spark is doing: **monitoring the cluster** and **adaptively partitioning the data** based on the cluster's available computational capacity.

So, the following code cell is where we actually utilize the AWS extension to Spark discussed above – we use the prefix `s3a://` to tell Spark that we'd like it to scan over a file that lives in an S3 bucket and partition the nodes in the cluster to process this data in a distributed manner (as efficiently as possible).

Feel free to browse the S3 bucket contents at the given URI (meaning, the files in the `dsan6000-unsc` bucket, with prefix `unsc-speeches`). You'll see that it contains a version of the dataset mentioned in the `README.md` file, **Hive-partitioned** by year, from `year=1946` to `year=2024`.

In [ ]:
s3_uri = "s3a://dsan6000-unsc/unsc-speeches"
unsc_sdf = (
  spark
  .read
  .option('header', 'true')
  .parquet(s3_uri)
)

26/11/05 19:49:04 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/ubuntu/dsan6000-hw07/.venv/lib/python3.14/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ubuntu/dsan6000-hw07/.venv/lib/python3.14/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.14/socket.py", line 725, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

26/11/05 19:49:19 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/11/05 19:49:34 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/11/05 19:49:49 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/11/05 19:50:04 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/11/05 19:50:19 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/11/05 19:50:34 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure th

In [7]:
unsc_sdf

DataFrame[speech_id: string, record_id: string, doc_name: string, meeting_num: bigint, year: bigint, month: bigint, day: bigint, topic: string, agenda: string, order: bigint, speaker: string, affiliation: string, position: string, president: boolean, secretary_general: boolean, procedural: boolean, word_count: bigint, speech: string, affiliation_cow: string, cow_ccode: double, permanent_member: bigint, elected_member: bigint, state: bigint, igo: bigint, un_org: bigint, ngo: bigint, year_meeting: bigint, month_meeting: bigint, day_meeting: bigint, meeting_num_meeting: bigint, closed: boolean, topic_meeting: string, agenda_meeting: string, agenda_categories: string, pres_name: string, pres_country: string, speeches: boolean, total_word_count: double, outcome: string, record: string, record_url: string, RES: string, RES_url: string, PRST: string, PRST_url: string, _merge: string]

In [9]:
first_speech = unsc_sdf.take(1)

In [11]:
type(first_speech)

list

In [12]:
type(first_speech[0])

pyspark.sql.types.Row

In [13]:
first_speech[0]

Row(speech_id='SC00001-01-001', record_id='SC00001-01', doc_name='S_PV.1', meeting_num=1, year=1946, month=1, day=17, topic='Opening of the meeting by the Temporary Chairman;Presentation of the section of the Report of the Preparatory Commission which relates to the Security Council;Adoption of rule 9 of the provisional rules of procedure providing a method for selecting the President;Selection of the President of the Security Council', agenda='Opening of the meeting by the temporary  Chairman; Presentation of the section of the Report of  the Preparatory Commission which relates  to the Security Council; Adoption of rule 9 of the provisional rules of procedure providing a method for selecting the President; Selection of the President of the Security Council; Adoption of the agenda; Adoption of provisional rules of procedure', order=1, speaker='Mr. N. J. O. MAKIN', affiliation='Australia', position=None, president=True, secretary_general=False, procedural=True, word_count=35, speech='I

In [14]:
first_speech[0].asDict()

{'speech_id': 'SC00001-01-001',
 'record_id': 'SC00001-01',
 'doc_name': 'S_PV.1',
 'meeting_num': 1,
 'year': 1946,
 'month': 1,
 'day': 17,
 'topic': 'Opening of the meeting by the Temporary Chairman;Presentation of the section of the Report of the Preparatory Commission which relates to the Security Council;Adoption of rule 9 of the provisional rules of procedure providing a method for selecting the President;Selection of the President of the Security Council',
 'agenda': 'Opening of the meeting by the temporary  Chairman; Presentation of the section of the Report of  the Preparatory Commission which relates  to the Security Council; Adoption of rule 9 of the provisional rules of procedure providing a method for selecting the President; Selection of the President of the Security Council; Adoption of the agenda; Adoption of provisional rules of procedure',
 'order': 1,
 'speaker': 'Mr. N. J. O. MAKIN',
 'affiliation': 'Australia',
 'position': None,
 'president': True,
 'secretary_ge

In [ ]:
unsc_sdf.rdd.getNumPartitions()